### start improving our rag systme in terms of retrieval capabilities
running multiple retrieval processed against different kidns of metrics
current vector db -> cosine similarity retrieval which retursn contexctually similar things

New: Lexical search
Trying to match keywords.....

We might retrieve some data from contextual search but if the search is extrmeely keyword focused 
e.g if we are looking for a specific pyuthon lib we might be looking by a specific method 

we will us qdrant built in mechs

### import dependencies

In [57]:
import openai
import pandas as pd
from qdrant_client import QdrantClient
from qdrant_client import models
from qdrant_client.models import VectorParams, Distance, SparseVectorParams, Modifier, PayloadSchemaType, PointStruct, Document, Prefetch, FusionQuery


### Create Qdrant Collection for Hybrid Search

In [58]:
qdrant_client = QdrantClient(url="http://localhost:6333")

In [59]:
# qdrant_client.create_collection(
#     collection_name="Amazon-items-collection-01-hybrid-search",
#     vectors_config={
#         "text-embedding-3-small": VectorParams(
#             size=1536,
#             distance=Distance.COSINE
#         )
#     },
#     sparse_vectors_config={
#         "bm25": SparseVectorParams(
#             modifier=Modifier.IDF
#         )
#     }
# )

In [60]:
# multiple queries against parent asin field
# we have one where we retrieve additional info price, image url
qdrant_client.create_payload_index(
    collection_name="Amazon-items-collection-01-hybrid-search",
    field_name="parent_asin",
    field_type=PayloadSchemaType.KEYWORD
)

UpdateResult(operation_id=10, status=<UpdateStatus.COMPLETED: 'completed'>)

### Embedding functions 

In [61]:
def get_embeddings_batch(text_list, model="text-embedding-3-small", batch_size=100):
    if len(text_list) <= batch_size:
        response = openai.embeddings.create(
            input=text_list,
            model=model
        )
        return [item.embedding for item in response.data]
    
    all_embeddings = []
    counter = 1
    for i in range(0, len(text_list), batch_size):
        batch = text_list[i:i+batch_size]
        response = openai.embeddings.create(
            input=batch,
            model=model
        )
        all_embeddings.extend([item.embedding for item in response.data])
        print(f"Processed {counter * batch_size} of {len(text_list)}")
        counter += 1
    return all_embeddings

In [62]:
df_items = pd.read_json("../../data/meta_Electronics_2022_2023_with_category_ratings_100_sample_1000.jsonl", orient="records", lines=True)
print(len(df_items))
df_items.head()

1000


,main_category,title,average_rating,rating_number,features,description,price,images,videos,store,categories,details,parent_asin,bought_together,subtitle,author
0,AMAZON FASHION,"Bosttor Bluetooth Beanie Hat with Light, Headl...",4.5,2342,"[100% Acrylic, Elastic closure, Hand Wash Only...",[],26.99,[{'thumb': 'https://m.media-amazon.com/images/...,[{'title': 'Beanie Hat with Bluetooth Headphon...,Bosttor,"[Electronics, Headphones, Earbuds & Accessorie...","{'Department': 'unisex-adult', 'Date First Ava...",B0BH41HYFZ,NaN,NaN,NaN
1,All Electronics,"Aceele USB and USB C to Ethernet Adapter, 3.3f...",4.2,148,[【USB or USB C to Ethernet Adapter】The Etherne...,[],14.99,[{'thumb': 'https://m.media-amazon.com/images/...,[],Aceele,"[Electronics, Computers & Accessories, Network...",{'Product Dimensions': '3.54 x 0.98 x 0.71 inc...,B0BKPB2YQ9,NaN,NaN,NaN
2,All Electronics,HDMI Switch 3 in 1 Out 4K UHD HDMI Switcher Sp...,4.2,3331,[📍3-Port HDMI Switch: This aluminum HDMI switc...,[],18.98,[{'thumb': 'https://m.media-amazon.com/images/...,[{'title': 'Darren reviews VWRHAR 3-1 splitter...,VWRHar,"[Electronics, Home Audio, Home Audio Accessori...",{'Package Dimensions': '6.1 x 4.21 x 0.94 inch...,B09MM5QT3R,NaN,NaN,NaN
3,All Electronics,"Smart Watch,Ip67 Waterproof Bluetooth Smartwat...",3.4,125,[],[],NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[],Burxoe,[],{'Package Dimensions': '3.15 x 3.07 x 2.52 inc...,B09Q5TNDHY,NaN,NaN,NaN
4,Cell Phones & Accessories,"(3 Pack) Cute Airpod Case for Airpods 2&1,3D D...",4.7,335,[【Compatible Airpods 1/2 】This case designed f...,[1],11.99,[{'thumb': 'https://m.media-amazon.com/images/...,"[{'title': 'airpod cases', 'url': 'https://www...",UGUHY,"[Electronics, Headphones, Earbuds & Accessorie...",{'Package Dimensions': '7.44 x 4.61 x 1.57 inc...,B0BZJKX7MS,NaN,NaN,NaN


In [63]:
def preprocess_description(row):
    return f"{row['title']}: {' '.join(row['features'])}"

def extract_first_large_image(row):
    return row['images'][0].get("large", "")

df_items["preprocessed_description"] = df_items.apply(preprocess_description, axis=1)
df_items["image"] = df_items.apply(extract_first_large_image, axis=1)

In [64]:
# df_items.head()
df_data_to_embed = df_items[["preprocessed_description", "image", "rating_number", "price",  "average_rating", "parent_asin"]]
data_to_embed = df_data_to_embed.to_dict(orient="records")

# data_to_embed -> list of dictionaries


In [65]:
# from data_to_embed
text_to_embed = [item["preprocessed_description"] for item in data_to_embed]



In [66]:
embeddings = get_embeddings_batch(text_to_embed)

Processed 100 of 1000
Processed 200 of 1000
Processed 300 of 1000
Processed 400 of 1000
Processed 500 of 1000
Processed 600 of 1000
Processed 700 of 1000
Processed 800 of 1000
Processed 900 of 1000
Processed 1000 of 1000


In [67]:
pointstructs = []
i = 1
for embedding, data in zip(embeddings, data_to_embed):
    pointstructs.append(PointStruct(
        id=i,
        vector={
            "text-embedding-3-small": embedding,
            "bm25": Document( # we do not precalculate the BM25 vectors (this happens natively in qdrant)
                text=data["preprocessed_description"],
                model='qdrant/bm25'
            )
        },
        payload=data
    ))
    i += 1

In [68]:
qdrant_client.upsert(
    collection_name="Amazon-items-collection-01-hybrid-search",
    points=pointstructs[0:500],
    wait=True
)

UpdateResult(operation_id=11, status=<UpdateStatus.COMPLETED: 'completed'>)

In [69]:
qdrant_client.upsert(
    collection_name="Amazon-items-collection-01-hybrid-search",
    points=pointstructs[500:],
    wait=True
)

UpdateResult(operation_id=12, status=<UpdateStatus.COMPLETED: 'completed'>)

### Hybrid retrieval

In [ ]:
def get_embedding(text, model="text-embedding-3-small"):
    response = openai.embeddings.create(
        input=text,
        model=model
    )
    return response.data[0].embedding

def retrieve_data(query, k=5): # 5 most similar items to users query
    embedding = get_embedding(query) # so we are actually creating related vector here
    results = qdrant_client.query_points(
        collection_name="Amazon-items-collection-01-hybrid-search",
        prefetch=[
            Prefetch(
                query=embedding,
                using='text-embedding-3-small', #name of the vector in the collection
                limit=20
            ), 
            Prefetch(
                query=Document(text=query, model='qdrant/bm25'),
                using='bm25', #name of the vector in the collection
                limit=20
            )
        ],
        # can set specific weights on how much importance we put on dense embeddings and bm25 lexical retrievals
        query=FusionQuery(fusion='rrf'), # how you fuse the results....
        limit=k # once fused we are returning top k results
    )
    
    retrieved_context_ids = []
    retrieved_context = []
    similarity_scores = []
    retrieved_context_ratings = []
    
    for result in results.points:
        retrieved_context_ids.append(result.payload["parent_asin"])
        retrieved_context.append(result.payload.get("preprocessed_description"))
        similarity_scores.append(result.score)
        retrieved_context_ratings.append(result.payload.get("average_rating"))
    
    return {
        "retrieved_context_ids": retrieved_context_ids,
        "retrieved_context": retrieved_context,
        "similarity_scores": similarity_scores,
        "retrieved_context_ratings": retrieved_context_ratings
    }

In [71]:
results = retrieve_data("Can I get a tablet?", k=20)

### Hybrid Serach with weighted RRF

In [ ]:
def get_embedding(text, model="text-embedding-3-small"):
    response = openai.embeddings.create(
        input=text,
        model=model
    )
    return response.data[0].embedding

# how we process context
def process_context(context):
    formatted_context = ""
    for id, chunk, rating in zip(context["retrieved_context_ids"], context["retrieved_context"], context["retrieved_context_ratings"]):
        formatted_context += f"- ID: {id}, rating: {rating}, description: {chunk}\n"
    return formatted_context

### using instructor when we actually pass the context and tell instructor we want to receive a list
### the desecription is hte chunk and we are expecting the references to have the id of the retreived context and a description baed on the chunk

def retrieve_data(query, k=5): # 5 most similar items to users query
    embedding = get_embedding(query) # so we are actually creating related vector here
    results = qdrant_client.query_points(
        collection_name="Amazon-items-collection-01-hybrid-search",
        prefetch=[
            Prefetch(
                query=embedding,
                using='text-embedding-3-small', #name of the vector in the collection
                limit=20
            ), 
            Prefetch(
                query=Document(text=query, model='qdrant/bm25'),
                using='bm25', #name of the vector in the collection
                limit=20
            )
        ],
        # can set specific weights on how much importance we put on dense embeddings and bm25 lexical retrievals
        # in this case dense vector retrival is weighted as 3 times more important than bm25
        # if user queries are more keyword-based, you might want to increase the weight of bm25
        # if more contextual then you might want to increase the weight of the dense vector retrieval
        # weights can be added as params to the retrieve data function
        query=models.RrfQuery(rrf=models.Rrf(weights=[3,1])), # how you fuse the results....
        limit=k # once fused we are returning top k results
    )
    
    retrieved_context_ids = []
    retrieved_context = []
    similarity_scores = []
    retrieved_context_ratings = []
    
    for result in results.points:
        retrieved_context_ids.append(result.payload["parent_asin"])
        retrieved_context.append(result.payload.get("preprocessed_description"))
        similarity_scores.append(result.score)
        retrieved_context_ratings.append(result.payload.get("average_rating"))
    
    return {
        "retrieved_context_ids": retrieved_context_ids,
        "retrieved_context": retrieved_context,
        "similarity_scores": similarity_scores,
        "retrieved_context_ratings": retrieved_context_ratings
    }